# 7장 에이전트 평가

TinyAgent가 주어진 과제를 해결했는지 측정합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 7장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 선택

앞에서 사용한 Gemma 4 E4B의 내장 추론과 도구 호출을 사용합니다. 뒤의 모델 평가자 실습에서는 같은 모델이 답변을 평가하기도 합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b", think=True)

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. `Evaluator` 구현

평가 문항을 구성하는 `Benchmark`와 답변을 채점하는 `Evaluator`를 정의합니다.

In [ ]:
from dataclasses import dataclass
from typing import Callable

# 채점 함수의 타입 힌트
##  (prediction: str, example: dict) -> bool | float
Scorer = Callable[[str, dict], bool | float]

@dataclass
class Benchmark:
    name: str
    examples: list[dict]
    scorer: Callable

In [ ]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct, TinyAgent


class Evaluator:
    """벤치마크의 각 문항을 TinyAgent로 실행하고 결과를 집계한다."""

    def __init__(self, create_agent: Callable):
        """새 에이전트를 만드는 함수를 전달받아 초기화한다."""
        self.create_agent = create_agent

    def run(self, benchmark: Benchmark) -> dict:
        """벤치마크의 각 예제를 실행하고 결과를 채점한다."""

        # 각 예제를 실행하고 결과 수집
        results = []
        for example in benchmark.examples:
            agent = self.create_agent()
            prediction = agent.run(example["task"]) or ""
            passed = benchmark.scorer(prediction, example)
            results.append(
                {
                    "prediction": prediction,
                    "passed": passed,
                }
            )

        # 통과율 계산
        if results:
            pass_rate = sum(result["passed"] for result in results) / len(results)
        else:
            pass_rate = 0.0

        # 상세 결과와 전체 통과율 반환
        return {
            "name": benchmark.name,
            "pass_rate": pass_rate,
            "results": results,
        }


def create_agent():
    """새 TinyAgent 인스턴스를 생성한다."""
    return TinyAgent(
        llm=llm,
        memory=Memory(),
        tools=NativeTools(),
        planner=NativeReAct(),
    )

`create_agent`는 새 평가를 시작할 때마다 `TinyAgent` 인스턴스를 만듭니다. 이전 평가의 대화가 메모리에 남아 다음 답변에 영향을 주지 않게 합니다.

## 3. 정답과 정확히 일치하는지 검사

일치 여부로 채점하면 허용된 정답을 정확하게 출력해야 점수를 받습니다. 이 예제는 여러 분야의 객관식 문제로 구성된 MMLU-Pro의 문항을 사용합니다.

문항은 뇌하수체가 들어 있는 체강, 호모 에렉투스의 평균 두개강 용량, 무어의 이상적 공리주의에서 옳은 행위가 최대화하는 대상을 묻습니다. 모델에는 선택지의 문자 하나만 답하도록 지시합니다.

In [ ]:
import re

def exact_match_scorer(prediction: str, example: dict) -> bool:
    """정답과 예측이 일치하면 True, 다르면 False를 반환한다."""
    match = re.search(r"\b([A-J])\b", prediction.upper())
    return match.group(1) == example["expected"]

In [ ]:
# MMLU-Pro의 예제 세 개
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": """Which of the following is the body cavity that contains the pituitary gland?
A) Ventral B) Dorsal C) Buccal D) Thoracic E) Pericardial F) Abdominal G) Spinal H) Pelvic I) Pleural J) Cranial
Answer with only the letter.""",
            "expected": "J",
        },
        {
            "task": """What is the approximate mean cranial capacity of Homo erectus?
A) 1200 cc B) under 650 cc C) 1700 cc D) 1350 cc E) just under 1000 cc F) 1500 cc G) under 500 cc H) about 800 cc I) just over 1100 cc J) about 900 cc
Answer with only the letter.""",
            "expected": "E",
        },
        {
            "task": """	
According to Moore’s “ideal utilitarianism,” the right action is the one that brings about the greatest amount of:
A) wealth. B) virtue. C) fairness. D) pleasure. E) peace. F) justice. G) happiness. H) power. I) good. J) knowledge.
Answer with only the letter.""",
            "expected": "I",
        },
    ],
    scorer=exact_match_scorer,
)

In [ ]:
from rich import print

# 평가 실행
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

## 4. 프로그램으로 조건 검사

IFEval의 예를 사용해 답변이 지켜야 하는 여러 조건을 코드로 검사합니다. 같은 채점 방식으로 서로 다른 형식의 요구 사항을 평가할 수 있습니다.

첫 문항은 초등학교 놀이터 신설을 제안하는 노래를 열 문장 미만으로 쓰게 하고 문장부호 수를 검사합니다. 두 번째는 사진 액자 소개를 150단어 이하로 쓰게 하고 단어 수를 셉니다. 세 번째는 일본 여행 일정을 셰익스피어 문체로 쓰게 하고 쉼표가 없는지 검사합니다. 각 함수가 실제로 확인하는 조건과 프롬프트에 적힌 요구 사항을 비교해 보세요.

In [ ]:
def programmatic_scorer(prediction: str, example: dict) -> bool:
    """해당 문항의 검사 함수로 예측을 확인한다."""
    return example["check"](prediction)

In [ ]:
# IFEval의 예제 세 개
ifeval = Benchmark(
    name="IFeval",
    examples=[
        {
            "task": "Write me a funny song with less than 10 sentences for a proposal to build a new playground at my local elementary school.",
            "check": lambda text: sum(1 for c in text if c in ".!?") < 10,
        },
        {
            "task": "Write an ad copy for a new product, a digital photo frame that connects to your social media accounts and displays your photos. Respond with at most 150 words.",
            "check": lambda text: len(text.split()) <= 150,
        },
        {
            "task": "I am planning a trip to Japan, and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.",
            "check": lambda text: "," not in text,
        },
    ],
    scorer=programmatic_scorer,
)

In [ ]:
# 평가 실행
result = Evaluator(create_agent).run(ifeval)
print(result)

## 5. 언어 모델을 평가자로 사용하기(LLM-as-a-judge)

별도의 모델 호출로 답변의 품질을 채점할 수 있습니다. 더 큰 모델로 작은 모델의 응답을 평가하는 방법에도 적용할 수 있습니다.

이번에는 같은 주제의 서술형 문항에 답하게 합니다. 평가자에게 기대 답변과 실제 답변을 함께 주고, 0.0~1.0 사이의 점수 하나만 출력하도록 요청합니다.

In [ ]:
judge = LLM(
    model="gemini-3.1-flash-lite", 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai", 
    api_key="MY_API_KEY"
)

In [ ]:
def judge_scorer(prediction: str, example: dict) -> bool:
    """모델을 평가자로 사용하는 채점 함수이다."""
    prompt = f"""
Score the response from 0.0 to 1.0.

Expected: {example["expected"]}
Response: {prediction}

Reply with only a single number.
"""
    response = judge.generate([{"role": "user", "content": prompt}])
    score = float(response.content.strip().split()[0])
    return score

In [ ]:
# MMLU-Pro를 바탕으로 만든 예제 세 개
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": "Which body cavity contains the pituitary gland?",
            "expected": "the cranial cavity",
        },
        {
            "task": "What is the approximate mean cranial capacity of Homo erectus?",
            "expected": "just under 1000 cc",
        },
        {
            "task": "According to Moore's 'ideal utilitarianism,' the right action is the one that brings about the greatest amount of what?",
            "expected": "good",
        },
    ],
    scorer=judge_scorer,
)

In [ ]:
# 평가 실행
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

---

# 구현한 내용

6장의 에이전트를 대상으로 별도의 평가 코드를 작성했습니다. 평가 대상과 채점 코드를 따로 두면 에이전트를 고칠 때도 같은 기준으로 결과를 비교할 수 있습니다.

In [ ]:
from illustrated_agents.chapters.ch7 import what_we_built; what_we_built

# 다음 장

[8장](../chapter08/chapter08.ipynb)에서는 여러 에이전트의 협업을 실습합니다.